In [1]:
import pandas as pd
import ast
import pickle

In [2]:
drug_entities = pd.read_csv('../../../outputs/preclinical_db/v3/drug_entities.tsv', sep='\t',
                            dtype=str,
                            keep_default_na=False)
disease_entities = pd.read_csv('../../../outputs/preclinical_db/v3/disease_entities.tsv', sep='\t',
                               dtype=str,
                                 keep_default_na=False)
animal_entities = pd.read_csv('../../../outputs/preclinical_db/v3/animal_entities.tsv', sep='\t',
                              dtype=str,
                                keep_default_na=False)


In [3]:
# we need to keep only verified ids:

def create_tree_id_lut():
    mesh_desc_data = pd.read_csv("/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_descriptive_data.tsv",
                                 sep="\t")
    mesh_desc_data["tree_numbers"] = \
        mesh_desc_data["tree_numbers"].apply(lambda x: ast.literal_eval(x))
    tree_id_lut = {}
    
    for _, row in mesh_desc_data.iterrows():
        # print(row['mesh_id'])
        tree_id_lut[row['mesh_id']] = row['tree_numbers']
    return tree_id_lut


def verify_mesh_ids(df):
    tree_id_lut = create_tree_id_lut()
    mesh_data = pd.read_csv('/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_data.tsv',
                        sep='\t')
    mesh_data = mesh_data.set_index("mesh_id")

    def _verify_mesh_id(mesh_ids):
        if mesh_ids == "":
            return ""
        if mesh_ids.startswith("["):
            mesh_ids = ast.literal_eval(mesh_ids)
        else:
            return mesh_ids
        verified_mesh_ids = []
        for mesh_id in mesh_ids:
            verified = False
            if mesh_id not in tree_id_lut:
                pas = mesh_data.loc[mesh_id]['pharmacological_actions']
                pas = ast.literal_eval(pas)
                if len(pas) > 0:
                    verified = True
            else:
                mesh_tree_ids = tree_id_lut[mesh_id]
                for tree_id in mesh_tree_ids:
                    if tree_id.startswith("D"):
                        verified = True
            if verified:
                verified_mesh_ids.append(mesh_id)
        if len(verified_mesh_ids) == 0:
            return ""
        return str(verified_mesh_ids)
    
    df["mesh_id"] = df["mesh_id"].apply(_verify_mesh_id)
    

verify_mesh_ids(drug_entities)

In [4]:
# we need to keep only verified ids:

def load_UMLS_semantic_type_lut():
    with open('/mnt/share/kaichixie/A2H_preclinical_database/data/UMLS/UMLS_semantic_LUT.pkl', 'rb') as f:
        umls_semantic_lut = pickle.load(f)
    return umls_semantic_lut


def verify_umls_ids(df):
    umls_semantic_lut = load_UMLS_semantic_type_lut()
    allowed_semantic_types = ['T195', #Antibiotic,
                            'T200', # Clinical drugs
                            'T121'] # Pharmalogical substance

    def _verify_umls_id(umls_ids):
        if umls_ids == "":
            return ""
        if umls_ids.startswith("["):
            umls_ids = ast.literal_eval(umls_ids)
        else:
            return umls_ids
        verified_umls_ids = []
        for umls_id in umls_ids:
            verified = False
            s_types = umls_semantic_lut[umls_id]
            for s_t in s_types:
                if s_t in allowed_semantic_types:
                    verified = True
            if verified:
                verified_umls_ids.append(umls_id)
        if len(verified_umls_ids) == 0:
            return ""
        return str(verified_umls_ids)
    
    df["UMLS_id"] = df["UMLS_id"].apply(_verify_umls_id)
    

verify_umls_ids(drug_entities)

In [5]:
for c in ["mesh_id", "UMLS_id", "drugbank_id"]:
    drug_entities[c] = drug_entities[c].apply(lambda x: [x] if not x.startswith("[") and x != "" else x)
    drug_entities[c] = drug_entities[c].apply(lambda x: ast.literal_eval(x) if not isinstance(x, list) and x != "" else x)
drug_entities

,PCDB_id,drug_name,drug_synonyms,drugbank_id,mesh_id,UMLS_id
0,PCDB_DR1,Diazoxide,"['product containing diazoxide', '2h-1,2,4-ben...",[DB01119],[D003981],[C0012022]
1,PCDB_DR2,GDC-0152,"['gdc-0152', 'smac mimetic gdc-0152', 'iap ant...",[DB12380],,[C2981796]
2,PCDB_DR3,Pheniramine,"['maleate, pheniramine', 'bimaleate, phenirami...",[DB01620],[D010632],"[C0031408, C0887016]"
3,PCDB_DR4,BMS-777607,"['aslan002', 'bms-777607', 'n-(4-(2-amino-3-ch...",[DB12064],,[C2346911]
4,PCDB_DR5,Trimethoprim,['trimetoprim:susceptibilidad:punto temporal:a...,[DB00440],[D014295],[C0041041]
...,...,...,...,...,...,...
11748,PCDB_DR11749,TS-011,['ts-011'],,,[C1615056]
11749,PCDB_DR11750,nabi-hb,"['nabi hb', 'nabi-hb novaplus', 'nabi-hb']",,,[C0876102]
11750,PCDB_DR11751,gomesin,['gomesin'],,,[C0971163]
11751,PCDB_DR11752,brontictuzumab,"['immunoglobulin g2-lambda, anti-(homo sapiens...",,,[C4053715]


In [6]:
with open('/mnt/share/kaichixie/A2H_preclinical_database/data/mesh/mesh_synonyms_look_up_table.pkl', 'rb') as f:
    mesh_synonyms_lut = pickle.load(f)

with open('/mnt/share/kaichixie/A2H_preclinical_database/data/UMLS/UMLS_synonyms_look_up_table.pkl', 'rb') as f:
    umls_synonyms_lut = pickle.load(f)

with open('/mnt/share/kaichixie/A2H_preclinical_database/data/drugbank/drugbank_synonyms_look_up_table.pkl', 'rb') as f:
    drugbank_synonyms_lut = pickle.load(f)

def find_drug_synonyms(row):
    mesh_ids = row["mesh_id"] if row["mesh_id"] != "" else []
    umls_ids = row["UMLS_id"] if row["UMLS_id"] != "" else []
    drugbank_ids = row["drugbank_id"] if row["drugbank_id"] != "" else []
    synonyms = []
    for drugbank_id in drugbank_ids:
        if drugbank_id in drugbank_synonyms_lut:
            synonyms.extend(drugbank_synonyms_lut[drugbank_id])
    for mesh_id in mesh_ids:
        if mesh_id in mesh_synonyms_lut:
            synonyms.extend(mesh_synonyms_lut[mesh_id])
    for umls_id in umls_ids:
        if umls_id in umls_synonyms_lut:
            synonyms.extend(umls_synonyms_lut[umls_id])
    synonyms = [s.lower() for s in synonyms]
    unique = list(dict.fromkeys(synonyms))[:10]
    sorted_unique = sorted(unique, key=len)
    return sorted_unique

drug_entities["drug_synonyms"] = drug_entities.apply(find_drug_synonyms, axis=1)
drug_entities

,PCDB_id,drug_name,drug_synonyms,drugbank_id,mesh_id,UMLS_id
0,PCDB_DR1,Diazoxide,"[diazoxide, diazoxido, hyperstat, proglycem, d...",[DB01119],[D003981],[C0012022]
1,PCDB_DR2,GDC-0152,"[gdc-0152, smac mimetic gdc-0152, iap antagoni...",[DB12380],,[C2981796]
2,PCDB_DR3,Pheniramine,"[feniramina, pheniramine, pheniraminum, prophe...",[DB01620],[D010632],"[C0031408, C0887016]"
3,PCDB_DR4,BMS-777607,"[aslan002, bms777607, bms-777607, bms 777607, ...",[DB12064],,[C2346911]
4,PCDB_DR5,Trimethoprim,"[trimpex, proloprim, trimetoprima, trimethopri...",[DB00440],[D014295],[C0041041]
...,...,...,...,...,...,...
11748,PCDB_DR11749,TS-011,[ts-011],,,[C1615056]
11749,PCDB_DR11750,nabi-hb,"[nabi-hb, nabi hb, nabi-hb novaplus]",,,[C0876102]
11750,PCDB_DR11751,gomesin,[gomesin],,,[C0971163]
11751,PCDB_DR11752,brontictuzumab,"[brontictuzumab, immunoglobulin g2-lambda, ant...",,,[C4053715]


In [7]:
def clean_disease_synonyms(row):
    if row["mesh_synonyms"] == "":
        mesh_synonyms = []
    else:
        mesh_synonyms = ast.literal_eval(row["mesh_synonyms"])
    if row["diseaseontology_synonyms"] == "":
        do_synonyms = []
    else:
        do_synonyms = ast.literal_eval(row["diseaseontology_synonyms"])
    all_synonyms = mesh_synonyms + do_synonyms
    all_synonyms = [s.lower() for s in all_synonyms]
    unique = list(dict.fromkeys(all_synonyms))[:10]
    sorted_unique = sorted(unique, key=len)
    if len(sorted_unique) == 0:
        return ""
    return sorted_unique

disease_entities["disease_synonyms"] = disease_entities.apply(clean_disease_synonyms, axis=1)

In [8]:
disease_entities.drop(columns=["synonyms"], inplace=True)

disease_entities["disease_mesh_id"] = \
    disease_entities.apply(lambda row: [row["disease_mesh_id"]] if row["disease_mesh_id"] != "" else "", axis=1)

disease_entities["diseaseontology_ids"] = \
    disease_entities.apply(lambda row: [row["diseaseontology_ids"]] if (not row["diseaseontology_ids"].startswith('[')) else ast.literal_eval(row["diseaseontology_ids"]), axis=1)
disease_entities["diseaseontology_ids"] = \
    disease_entities["diseaseontology_ids"].apply(lambda x: "" if len(x) == 1 and x[0] == "" else x)
disease_entities.rename(columns={"disease_mesh_id": "mesh_id", "diseaseontology_ids": "diseaseontology_id"},
                        inplace=True)

In [9]:
def clean_animal_synonyms(row):
    if row["synonyms"] == "":
        mesh_synonyms = []
    else:
        mesh_synonyms = ast.literal_eval(row["synonyms"])
    all_synonyms = mesh_synonyms
    all_synonyms = [s.lower() for s in all_synonyms]
    unique = list(dict.fromkeys(all_synonyms))[:10]
    sorted_unique = sorted(unique, key=len)
    if len(sorted_unique) == 0:
        return ""
    return sorted_unique

animal_entities["animal_synonyms"] = animal_entities.apply(clean_animal_synonyms, axis=1)

In [10]:
animal_entities.drop(columns=["synonyms"], inplace=True)

In [14]:
# animal_entities.to_csv('../../../outputs/preclinical_db/v3/animal_entities_v2.tsv', sep='\t', index=False)
animal_entities.to_csv('../../../outputs/preclinical_db/v4/animal_entities.tsv', sep='\t', index=False)

In [15]:
# disease_entities.to_csv('../../../outputs/preclinical_db/v3/disease_entities_v2.tsv', sep='\t', index=False)
disease_entities.to_csv('../../../outputs/preclinical_db/v4/disease_entities.tsv', sep='\t', index=False)

In [16]:
# drug_entities.to_csv('../../../outputs/preclinical_db/v3/drug_entities_v2.tsv', sep='\t', index=False)
drug_entities.to_csv('../../../outputs/preclinical_db/v4/drug_entities.tsv', sep='\t', index=False)